# The Physics of Semantic Search: Deriving InfoNCE Gradients

When building retrieval-augmented generation (RAG) pipelines or designing memory for AI agents, we rely on vector embeddings to retrieve relevant context. But how does a neural network actually learn to map sentences into a dense mathematical space where "similar" concepts cluster together and "different" concepts push apart?

The engine behind this is **Contrastive Learning**, driven by the **InfoNCE (Normalized Temperature-scaled Cross-Entropy) Loss**.

This notebook proves exactly how query vectors update in space by comparing PyTorch's Autograd engine with our own manual mathematical derivation.

### The Mathematical Goal
We want to understand how the query vector ($q$) updates. The gradient $\nabla_q L$ dictates the physical forces acting on the embedding during training:
1. **The Pull:** The query is pulled *towards* the positive document $d_+$ scaled by $(p_+ - 1)$. 
2. **The Push:** The query is pushed *away* from every negative document $d_-$ scaled strictly by their false probability $p_i$.


In [12]:
import torch
import torch.nn.functional as F
torch.manual_seed(42)

d_model = 4   # Dimension of our embeddings (simplified for the proof)
batch_size = 5 # 1 positive document, 4 negative documents
tau = 0.1     # Temperature parameter to scale the logits

print(f"Embedding Dimension: {d_model} | Batch Size: {batch_size} | Temperature: {tau}")

# Initialize the Query vector. 
q = torch.randn(d_model, requires_grad=True)

# Initialize the Document vectors.
docs = torch.randn(batch_size, d_model)
d_pos = docs[0]

Embedding Dimension: 4 | Batch Size: 5 | Temperature: 0.1


## Step 1: The Forward Pass (PyTorch Autograd)
First, we calculate the standard forward pass. We compute the dot products between the query and all documents, scale them by the temperature ($\tau$), and apply Cross-Entropy Loss.

By calling `.backward()`, PyTorch's Autograd engine automatically calculates the exact gradient matrix for the query vector based on the computational graph.

In [13]:
# Calculate scaled similarities (logits) for the entire batch
similarities = torch.matmul(docs, q) / tau
loss = F.cross_entropy(similarities.unsqueeze(0), torch.tensor([0]))

# Trigger backpropagation to compute gradients
loss.backward()

# Store the Autograd result for validation later
pytorch_grad = q.grad.clone()

print(f"Loss computed: {loss.item():.4f}")

Loss computed: 0.0092


## Step 2: The Manual Pass (Mathematical Derivation)

Now, we bypass Autograd and manually implement the mathematical gradient equation:

$$\nabla_q L = \frac{1}{\tau} \left( (p_+ - 1) d_+ + \sum_{i-} p_i d_i \right)$$

Here we can see the literal mechanism of contrastive learning. The positive document exerts a pull based on how uncertain the model is $(p_+ - 1)$, while the negative documents exert a push proportional to the model's false confidence in them $(p_i)$.

In [14]:
# We use torch.no_grad() because we are manually calculating the math, not building a graph
with torch.no_grad():
    probs = F.softmax(similarities, dim=0)
    manual_grad = torch.zeros_like(q)
    manual_grad += (probs[0] - 1.0) * d_pos
    for i in range(1, batch_size):
        manual_grad += probs[i] * docs[i]
        
    # 3. Apply the final temperature scaling
    manual_grad /= tau

## Step 3: Validation

Finally, we assert that our manually derived physical forces perfectly match the gradients calculated by PyTorch's C++ backend.

In [15]:
print("--- Gradient of Query Vector (dq) ---")
print(f"PyTorch Autograd: {pytorch_grad.numpy()}")
print(f"Manual Math Deriv: {manual_grad.numpy()}")

assert torch.allclose(pytorch_grad, manual_grad, atol=1e-6), "Gradients do not match!"
print("\n Success: The mathematical derivation perfectly matches the automatic differentiation.")

--- Gradient of Query Vector (dq) ---
PyTorch Autograd: [-0.26831052 -0.01386968 -0.01441914  0.2042279 ]
Manual Math Deriv: [-0.26831052 -0.01386969 -0.01441915  0.2042279 ]

 Success: The mathematical derivation perfectly matches the automatic differentiation.


### **Conclusion: The Physical Mechanics of Semantic Memory**

By deriving the InfoNCE loss function, we have mathematically proven how large language models and RAG systems learn to organize knowledge. Instead of relying on rigid keyword matching, contrastive learning transforms document retrieval into a physics problem governed by two distinct forces in vector space:

* **The Pull (Targeting Truth):** The gradient $(p_+ - 1) d_+$ ensures the model actively pulls the query toward the correct document. Crucially, as the model's confidence approaches $100\%$ ($p_+ \approx 1$), this force drops to zero. This physically prevents the neural network from wasting computational energy over-optimizing easy matches.
* **The Push (Repelling Distractors):** The gradient $\sum p_i d_i$ dictates that the query is actively pushed away from irrelevant documents. The magnitude of this push is strictly proportional to the model's false confidence ($p_i$). It barely reacts to obvious mismatches, but it exerts massive force against deceptive, "hard" negatives.
* **The Temperature Scaling ($\tau$):** Acting as the thermodynamic controller of the system, $\tau$ dictates the strictness of these forces. A lower temperature sharpens the probability distribution, forcing the model to aggressively penalize even minor similarities to incorrect documents.

**The Engineering Impact**
For data scientists and AI engineers, understanding this calculus is the difference between an agent that hallucinates and one that retrieves flawless context. By mastering these exact mathematical mechanics, we can fine-tune embedding models to navigate massive, highly ambiguous enterprise datasets—ensuring that when an AI reaches into its memory, it retrieves exactly the right data.